# Custom User-based Model
The present notebooks aims at creating a UserBased class that inherits from the Algobase class (surprise package) and that can be customized with various similarity metrics, peer groups and score aggregation functions. 

In [15]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# standard library imports
# -- add new imports here --

# third parties imports
import numpy as np 
import pandas as pd
from surprise import AlgoBase
# -- add new imports here --
from surprise import KNNWithMeans
import heapq

from surprise import AlgoBase
from surprise import PredictionImpossible

# local imports
from constants import Constant as C
from loaders import load_ratings
# -- add new imports here --

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 1. Loading Data
Prepare a dataset in order to help implementing a user-based recommender system

In [11]:
from pathlib import Path

C.DATA_PATH = Path("data/test")
C.CONTENT_PATH = C.DATA_PATH / "content"
C.EVIDENCE_PATH = C.DATA_PATH / "evidence"
C.EVALUATION_PATH = C.CONTENT_PATH.parent / "evaluation"

print("Dataset utilisé :", C.DATA_PATH)
print("Fichier ratings existe ?", (C.EVIDENCE_PATH / C.RATINGS_FILENAME).exists())

Dataset utilisé : data\test
Fichier ratings existe ? True


In [12]:
# -- load data, build trainset and anti testset --

data = load_ratings(surprise_format=True)

trainset = data.build_full_trainset()

anti_testset = trainset.build_anti_testset()

# 2. Explore Surprise's user-based algorithm
Displays user-based predictions and similarity matrix on the test dataset using the KNNWithMeans class

In [ ]:
# -- using surprise's user-based algorithm, explore the impact of different parameters and displays predictions --

sim_options = { # dictionnary containt the key-value 
    "name": "msd", #msd = mean square distance
    "user_based": True,
    "min_support": 3
}

algo = KNNWithMeans( # algorithm to compute User-Based or Items-based prediction
    k=3,
    min_k=2,
    sim_options=sim_options
)

algo.fit(trainset)

prediction = algo.predict(uid=11, iid=364) # uid user-id, iid item-id

prediction

Computing the msd similarity matrix...
Done computing similarity matrix.


Prediction(uid=11, iid=364, r_ui=None, est=2.49203431372549, details={'actual_k': 2, 'was_impossible': False})

In [14]:
# -- Playing with KNN --

# 1. Predictions on the anti-testset
predictions = algo.test(anti_testset)

print("First 30 predictions:")
for pred in predictions[:30]:
    print(pred)


# 2. Impact of min_k
for min_k_value in [1, 2, 3]:
    
    sim_options = {
        "name": "msd",
        "user_based": True,
        "min_support": 3
    }

    algo = KNNWithMeans(
        k=3,
        min_k=min_k_value,
        sim_options=sim_options
    )

    algo.fit(trainset)

    predictions = algo.test(anti_testset)

    print("\nmin_k =", min_k_value)
    for pred in predictions[:30]:
        print(pred)


# 3. Impact of min_support
for min_support_value in [1, 2, 3]:
    
    sim_options = {
        "name": "msd",
        "user_based": True,
        "min_support": min_support_value
    }

    algo = KNNWithMeans(
        k=3,
        min_k=2,
        sim_options=sim_options
    )

    algo.fit(trainset)

    predictions = algo.test(anti_testset)

    print("\nmin_support =", min_support_value)
    for pred in predictions[:30]:
        print(pred)


# 4. Similarity matrix
algo.sim[:5, :5]

First 30 predictions:
user: 11         item: 1214       r_ui = 3.13   est = 3.17   {'actual_k': 1, 'was_impossible': False}
user: 11         item: 364        r_ui = 3.13   est = 2.49   {'actual_k': 2, 'was_impossible': False}
user: 11         item: 4308       r_ui = 3.13   est = 3.17   {'actual_k': 1, 'was_impossible': False}
user: 11         item: 527        r_ui = 3.13   est = 3.90   {'actual_k': 2, 'was_impossible': False}
user: 13         item: 1997       r_ui = 3.13   est = 2.80   {'actual_k': 0, 'was_impossible': False}
user: 13         item: 4993       r_ui = 3.13   est = 2.80   {'actual_k': 1, 'was_impossible': False}
user: 13         item: 2700       r_ui = 3.13   est = 2.80   {'actual_k': 0, 'was_impossible': False}
user: 13         item: 1721       r_ui = 3.13   est = 2.80   {'actual_k': 1, 'was_impossible': False}
user: 13         item: 527        r_ui = 3.13   est = 2.80   {'actual_k': 1, 'was_impossible': False}
user: 17         item: 2028       r_ui = 3.13   est = 3.81  

array([[1.        , 0.        , 0.24615385, 0.        , 0.43243243],
       [0.        , 1.        , 0.        , 0.        , 0.17094017],
       [0.24615385, 0.        , 1.        , 0.        , 0.53333333],
       [0.        , 0.        , 0.        , 1.        , 0.        ],
       [0.43243243, 0.17094017, 0.53333333, 0.        , 1.        ]])

# 3. Implement and explore a customizable user-based algorithm
Create a self-made user-based algorithm allowing to customize the similarity metric, peer group calculation and aggregation function

In [22]:
class UserBased(AlgoBase):
    def __init__(self, k=3, min_k=1, sim_options={}, **kwargs):
        AlgoBase.__init__(self, sim_options=sim_options, **kwargs)
        self.k = k
        self.min_k = min_k

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)

        self.compute_rating_matrix()
        self.compute_similarity_matrix()

        self.mean_ratings = []

        for u in range(self.trainset.n_users):
            ratings_u = [rating for (_, rating) in self.trainset.ur[u]]
            mean_u = np.mean(ratings_u)
            self.mean_ratings.append(mean_u)

        return self

    def estimate(self, u, i):
        if not (self.trainset.knows_user(u) and self.trainset.knows_item(i)):
            raise PredictionImpossible("User and/or item is unknown.")

        estimate = self.mean_ratings[u]

        neighbors = []

        for neighbor_u, rating in self.trainset.ir[i]:
            if neighbor_u != u:
                similarity = self.sim[u, neighbor_u]

                if similarity > 0:
                    neighbors.append((similarity, neighbor_u, rating))

        top_neighbors = heapq.nlargest(self.k, neighbors, key=lambda x: x[0])

        numerator = 0
        denominator = 0
        actual_k = 0

        for similarity, neighbor_u, rating in top_neighbors:
            numerator += similarity * (rating - self.mean_ratings[neighbor_u])
            denominator += similarity
            actual_k += 1

        if actual_k >= self.min_k and denominator != 0:
            estimate += numerator / denominator

        details = {
            "actual_k": actual_k
        }

        return estimate, details

    def compute_rating_matrix(self):
        m = self.trainset.n_users
        n = self.trainset.n_items

        self.ratings_matrix = np.empty((m, n))
        self.ratings_matrix[:] = np.nan

        for u in range(m):
            for i, rating in self.trainset.ur[u]:
                self.ratings_matrix[u, i] = rating

    def compute_similarity_matrix(self):
        m = self.trainset.n_users

        self.sim = np.eye(m)

        min_support = self.sim_options.get("min_support", 1)
        similarity_name = self.sim_options.get("name", "msd")

        for u in range(m):
            for v in range(u + 1, m):

                row_u = self.ratings_matrix[u]
                row_v = self.ratings_matrix[v]

                rated_by_u = ~np.isnan(row_u)
                rated_by_v = ~np.isnan(row_v)

                common_ratings = rated_by_u & rated_by_v
                support = np.sum(common_ratings)

                if support >= min_support:

                    if similarity_name == "msd":
                        diff = row_u[common_ratings] - row_v[common_ratings]
                        msd = np.mean(diff ** 2)
                        similarity = 1 / (1 + msd)

                    elif similarity_name == "jacard":
                        intersection = np.sum(rated_by_u & rated_by_v)
                        union = np.sum(rated_by_u | rated_by_v)

                        if union != 0:
                            similarity = intersection / union
                        else:
                            similarity = 0

                    else:
                        raise ValueError(f"Unknown similarity metric: {similarity_name}")

                else:
                    similarity = 0

                self.sim[u, v] = similarity
                self.sim[v, u] = similarity

# 4. Compare KNNWithMeans with UserBased
Try to replicate KNNWithMeans with your self-made UserBased and check that outcomes are identical

In [23]:
# -- assert that predictions are the same with different sim_options --

sim_options = {
    "name": "msd",
    "user_based": True,
    "min_support": 3
}

# Surprise model
algo_surprise = KNNWithMeans(
    k=3,
    min_k=2,
    sim_options=sim_options
)

algo_surprise.fit(trainset)
predictions_surprise = algo_surprise.test(anti_testset)


# Custom model
algo_custom = UserBased(
    k=3,
    min_k=2,
    sim_options=sim_options
)

algo_custom.fit(trainset)
predictions_custom = algo_custom.test(anti_testset)


# Compare first 30 predictions
for pred_surprise, pred_custom in zip(predictions_surprise[:30], predictions_custom[:30]):
    print(pred_surprise.est, pred_custom.est)
    assert round(pred_surprise.est, 6) == round(pred_custom.est, 6)

print("The first 30 predictions are identical.")

Computing the msd similarity matrix...
Done computing similarity matrix.
3.1666666666666665 3.1666666666666665
2.49203431372549 2.49203431372549
3.1666666666666665 3.1666666666666665
3.898897058823529 3.898897058823529
2.8 2.8
2.8 2.8
2.8 2.8
2.8 2.8
2.8 2.8
3.8125 3.8125
4.128289473684211 4.128289473684211
3.25 3.25
3.25 3.25
3.5 3.5
3.5 3.5
3.5 3.5
3.5 3.5
3.5 3.5
3.5 3.5
3.5 3.5
3.5 3.5
2.782649253731343 2.782649253731343
2.349813432835821 2.349813432835821
4.666666666666667 4.666666666666667
4.666666666666667 4.666666666666667
4.666666666666667 4.666666666666667
4.666666666666667 4.666666666666667
4.666666666666667 4.666666666666667
4.666666666666667 4.666666666666667
4.666666666666667 4.666666666666667
The first 30 predictions are identical.


# 5. Compare MSD and Jacard
Compare predictions made with MSD similarity and Jacard similarity


In [24]:
# -- compare predictions made with MSD similarity and Jacard similarity --
# -- use UserBased with Jacard similarity --

sim_options = {
    "name": "jacard",
    "user_based": True,
    "min_support": 3
}

algo_jacard = UserBased(
    k=3,
    min_k=2,
    sim_options=sim_options
)

algo_jacard.fit(trainset)

predictions_jacard = algo_jacard.test(anti_testset)

predictions_jacard[:30]

# -- compare MSD and Jacard predictions --

sim_options_msd = {
    "name": "msd",
    "user_based": True,
    "min_support": 3
}

algo_msd = UserBased(
    k=3,
    min_k=2,
    sim_options=sim_options_msd
)

algo_msd.fit(trainset)
predictions_msd = algo_msd.test(anti_testset)


sim_options_jacard = {
    "name": "jacard",
    "user_based": True,
    "min_support": 3
}

algo_jacard = UserBased(
    k=3,
    min_k=2,
    sim_options=sim_options_jacard
)

algo_jacard.fit(trainset)
predictions_jacard = algo_jacard.test(anti_testset)


for pred_msd, pred_jacard in zip(predictions_msd[:30], predictions_jacard[:30]):
    print(
        "user:", pred_msd.uid,
        "| item:", pred_msd.iid,
        "| msd:", round(pred_msd.est, 3),
        "| jacard:", round(pred_jacard.est, 3),
        "| actual_k jacard:", pred_jacard.details["actual_k"]
    )

user: 11 | item: 1214 | msd: 3.167 | jacard: 3.167 | actual_k jacard: 1
user: 11 | item: 364 | msd: 2.492 | jacard: 2.167 | actual_k jacard: 2
user: 11 | item: 4308 | msd: 3.167 | jacard: 3.167 | actual_k jacard: 1
user: 11 | item: 527 | msd: 3.899 | jacard: 4.056 | actual_k jacard: 2
user: 13 | item: 1997 | msd: 2.8 | jacard: 2.8 | actual_k jacard: 0
user: 13 | item: 4993 | msd: 2.8 | jacard: 2.8 | actual_k jacard: 1
user: 13 | item: 2700 | msd: 2.8 | jacard: 2.8 | actual_k jacard: 0
user: 13 | item: 1721 | msd: 2.8 | jacard: 2.8 | actual_k jacard: 1
user: 13 | item: 527 | msd: 2.8 | jacard: 2.8 | actual_k jacard: 1
user: 17 | item: 2028 | msd: 3.812 | jacard: 3.907 | actual_k jacard: 2
user: 17 | item: 4993 | msd: 4.128 | jacard: 4.463 | actual_k jacard: 2
user: 17 | item: 1214 | msd: 3.25 | jacard: 3.25 | actual_k jacard: 1
user: 17 | item: 4308 | msd: 3.25 | jacard: 3.25 | actual_k jacard: 1
user: 19 | item: 1997 | msd: 3.5 | jacard: 3.5 | actual_k jacard: 0
user: 19 | item: 2028 |

In [25]:
print("MSD similarity matrix:")
print(algo_msd.sim[:5, :5])

print("\nJacard similarity matrix:")
print(algo_jacard.sim[:5, :5])

MSD similarity matrix:
[[1.         0.         0.24615385 0.         0.43243243]
 [0.         1.         0.         0.         0.17094017]
 [0.24615385 0.         1.         0.         0.53333333]
 [0.         0.         0.         1.         0.        ]
 [0.43243243 0.17094017 0.53333333 0.         1.        ]]

Jacard similarity matrix:
[[1.    0.    0.5   0.    0.4  ]
 [0.    1.    0.    0.    0.625]
 [0.5   0.    1.    0.    0.4  ]
 [0.    0.    0.    1.    0.   ]
 [0.4   0.625 0.4   0.    1.   ]]
